In [ ]:
import torch
import torchvision

import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

from derivative import gauge_derivative
from gaussian_blur import gaussian_blur
from frames import gauge_frame_hessian, gauge_frame_structure_tensor

In [ ]:
sigma = 2.0
with cbook.get_sample_data("grace_hopper.jpg") as f:
    image = torchvision.transforms.functional.pil_to_tensor(Image.open(f).convert("L"))[0] / 255
blurred = gaussian_blur(image, sigma=sigma)

fig, axes = plt.subplots(1, 2, figsize=(7, 4))
for ax, a, title in zip(axes, (image, blurred), ("original", f"smoothed (sigma = {sigma})")):
    ax.imshow(a, cmap="gray")
    ax.set_title(title, fontsize=10)

In [ ]:
frame_st = gauge_frame_structure_tensor(blurred, sigma=1.0)
frame_h = gauge_frame_hessian(blurred)

In [ ]:
FRAME_COLORS = ("#2a78d6", "#eb6834")
def show_gauge_frame(ax, image, frame, r0, r1, c0, c1, step):
    """Draw the gauge frame over rows r0:r1 and columns c0:c1, every step-th pixel, onto ax."""
    ax.imshow(image[r0:r1, c0:c1], cmap="gray", extent=(c0, c1, r1, r0))
    ys, xs = torch.meshgrid(torch.arange(r0, r1, step), torch.arange(c0, c1, step), indexing="ij")
    for i, color in enumerate(FRAME_COLORS):
        v = frame[r0:r1:step, c0:c1:step, :, i]
        ax.quiver(
            xs, ys, 
            v[..., 1], v[..., 0], 
            color=color, pivot="mid", angles="xy",
            headwidth=0, headlength=0, headaxislength=0, 
            scale=40, width=0.004
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 5))
fig.suptitle("Gauge frames obtained by taking eigenbasis of...", fontsize=11)
axes[0].set_title("Structure tensor", fontsize=10)
axes[1].set_title("Hessian", fontsize=10)
show_gauge_frame(axes[0], blurred, frame_st, 120, 350, 160, 360, 8)
show_gauge_frame(axes[1], blurred, frame_h, 120, 350, 160, 360, 8)
fig.tight_layout()

In [ ]:
def show_gauge_derivative(ax, image, frame, signature, quantile=0.99):
    field = gauge_derivative(image, frame, signature)
    v = field.abs().quantile(quantile).item()
    im = ax.imshow(field.abs(), cmap="viridis", vmin=0, vmax=v)
    ax.set_title(f"signature {signature}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    bar = ax.figure.colorbar(im, ax=ax, shrink=0.5)
    bar.outline.set_visible(False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 5))
for ax, signature in zip(axes, ([0], [1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("first-order gauge derivatives", fontsize=11)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, signature in zip(axes.flat, ([0, 0], [0, 1], [1, 1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("second-order gauge derivatives", fontsize=11)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, signature in zip(axes.flat, ([0, 0, 0], [0, 0, 1], [0, 1, 1], [1, 1, 1])):
    show_gauge_derivative(ax, blurred, frame_st, signature)
fig.suptitle("third-order gauge derivatives", fontsize=11)
fig.tight_layout()